# RAG Basics

## Learning goals

- understand what retrieval-augmented generation means in this repository
- inspect sentence chunking and local vector retrieval step by step
- compare the default TF-IDF retriever with the optional FAISS retriever path
- observe where naive RAG performs well and where it starts to fail


## Concept explanation

Before running any experiment, verify the Python executable and runtime profile. This matters on laptops, remote Jupyter servers, and DGX boxes because the notebook should point at the same `uv` environment that owns the dependencies.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## What is RAG

RAG combines retrieval and answer generation. In this project the retriever selects small document chunks, and the baseline answer builder stitches together evidence from those chunks. The key idea is simple: answer from retrieved context instead of pretending the model already knows everything.


In [ ]:
import pandas as pd

from src.ingestion import load_documents

documents = load_documents()
document_frame = pd.DataFrame(
    [
        {'doc_id': item['doc_id'], 'source': item['source'], 'characters': len(item['text'])}
        for item in documents
    ]
)
document_frame

## What is document chunking

Retrievers usually work better on chunks than on full documents. Smaller chunks narrow the evidence window and make it easier to inspect why a result ranked well. This repository uses sentence-based chunking so each chunk stays readable in the notebook.


In [ ]:
from src.ingestion import ingest_documents

chunks = ingest_documents(persist=False)
chunk_frame = pd.DataFrame(chunks)
chunk_frame[['doc_id', 'chunk_id', 'source', 'text']].head(10)

## What is embedding retrieval

The default retriever uses a TF-IDF plus lexical-overlap hybrid because it is reproducible and light. The repository also now includes an optional FAISS plus sentence-transformers retriever for dense retrieval experiments. The cell below tries both paths and reports which one is available in the current environment.


In [ ]:
from src.ingestion import build_demo_index

tfidf_retriever = build_demo_index(persist=False, backend='tfidf')
faiss_retriever = build_demo_index(persist=False, backend='faiss')
retriever_summary = pd.DataFrame(
    [
        {'backend': 'tfidf', 'class_name': type(tfidf_retriever).__name__},
        {'backend': 'faiss', 'class_name': type(faiss_retriever).__name__},
    ]
)
retriever_summary

## Implementation

Once the index exists, retrieval becomes a scored ranking problem. We query the TF-IDF retriever first because that path is always available, then optionally inspect the FAISS path if the environment has the dense retrieval dependencies.


In [ ]:
retrieval_query = 'What are the goals of the workspace policy refresh?'
tfidf_results = pd.DataFrame(tfidf_retriever.search(retrieval_query, top_k=4))
faiss_results = pd.DataFrame(faiss_retriever.search(retrieval_query, top_k=4))

print('TF-IDF top results')
display(tfidf_results[['chunk_id', 'source', 'score', 'text']])
print('FAISS path top results')
display(faiss_results[['chunk_id', 'source', 'score', 'text']] if not faiss_results.empty else faiss_results)

## Simple QA

The baseline RAG path retrieves context and then synthesizes a short answer without planning, tools, or grounding verification. That makes it a strong teaching baseline because you can clearly see what information retrieval alone can and cannot do.


In [ ]:
from src.workflow import run_baseline_rag

baseline_result = run_baseline_rag(
    'What are the main goals of the workspace policy refresh?',
    retriever=tfidf_retriever,
)

pd.Series(
    {
        'final_status': baseline_result['final_status'],
        'final_answer': baseline_result['final_answer'],
        'retrieved_sources': [item['source'] for item in baseline_result['retrieved_docs']],
    }
)

## Experiment

A useful experiment is to compare an easy lookup question with a question that really wants planning or tools. The contrast shows where baseline RAG is enough and where an agentic workflow adds value.


In [ ]:
experiment_questions = [
    'When does the organization-wide rollout begin?',
    'How many days are in the pilot window?',
]
experiment_rows = []
for question in experiment_questions:
    result = run_baseline_rag(question, retriever=tfidf_retriever)
    experiment_rows.append(
        {
            'question': question,
            'final_status': result['final_status'],
            'retrieved_sources': ', '.join(sorted({doc['source'] for doc in result['retrieved_docs']})),
            'answer': result['final_answer'],
        }
    )

pd.DataFrame(experiment_rows)

## Result analysis

The table below turns the experiment into a quick reading exercise. Notice how the rollout-date question matches a single document cleanly, while the pilot-window question depends on date reasoning that the baseline path only approximates through retrieved text.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'backend': 'TF-IDF baseline',
            'strength': 'Fast, deterministic, easy to inspect',
            'limitation': 'No planning, tool use, or abstention logic',
        },
        {
            'backend': 'Optional FAISS path',
            'strength': 'Dense similarity can surface semantically close evidence',
            'limitation': 'Needs extra dependencies and more setup',
        },
    ]
)
analysis_frame

## Takeaways

### Limitations of naive RAG

- Retrieval alone does not decide whether a question needs planning or tools.
- Baseline answers do not verify grounding before responding.
- Dense retrieval can help, but architecture still matters more than swapping the vector backend.
- This notebook gives us a control condition for the later agentic notebooks.
